In [10]:
#to set the earthdata env
import os
import getpass
import earthaccess

os.environ["EARTHDATA_USERNAME"] = input("Earthdata username: ").strip()
os.environ["EARTHDATA_PASSWORD"] = getpass.getpass("Earthdata password: ")

earthaccess.login(strategy="environment")


Earthdata username:  ralfriedel
Earthdata password:  ········


In [14]:
# ================================================================
# PACE Level-3 starter: regional pull + simple plots
# (finds input/output directories for the files specified below)
# ================================================================

import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import earthaccess
import xarray as xr


# ----------------------------
# 1) Diagnose runtime paths
# ----------------------------
print("CWD:", Path.cwd())
print("Home:", Path.home())

for p in [Path("/"), Path.cwd(), Path.home()]:
    try:
        kids = list(p.iterdir())
        print(f"\nListing {p} (first 30):")
        for k in kids[:30]:
            print(" -", k)
    except Exception as e:
        print(f"\nCould not list {p}: {e}")


# ----------------------------
# 2) Find the CSV
# ----------------------------
CSV_NAME = "combined_hypoxia_data.csv"   # change to lsu_hypoxia_2024.csv if needed

def find_input_csv() -> Path:
    roots = [
        Path("/data"),
        Path("/mnt/data"),
        Path("/workspace"),
        Path("/work"),
        Path("/home"),
        Path.cwd(),
    ]

    for r in roots:
        try:
            if (r / CSV_NAME).exists():
                return r / CSV_NAME
            if (r / "data" / CSV_NAME).exists():
                return r / "data" / CSV_NAME
        except Exception:
            pass

    for r in [Path.cwd(), Path.home()]:
        try:
            for cand in r.rglob(CSV_NAME):
                return cand
        except Exception:
            pass

    raise FileNotFoundError(f"Could not locate {CSV_NAME}")

INPUT_CSV = find_input_csv()
print("\nFOUND INPUT_CSV:", INPUT_CSV.resolve())


# ----------------------------
# 3) Choose a writable output dir
# ----------------------------
def first_writable_dir(candidates):
    for d in candidates:
        try:
            d = Path(d)
            d.mkdir(parents=True, exist_ok=True)
            test = d / ".write_test"
            test.write_text("ok")
            test.unlink()
            return d
        except Exception:
            continue
    return None

OUT_CANDIDATES = [
    Path("/output"),
    Path("/mnt/data/output"),
    Path.cwd() / "output",
    Path.cwd(),
]

OUTPUT_DIR = first_writable_dir(OUT_CANDIDATES)
if OUTPUT_DIR is None:
    raise RuntimeError("No writable output directory found.")

print("USING OUTPUT_DIR:", OUTPUT_DIR.resolve())


# ================================================================
# Load field data
# ================================================================
field = pd.read_csv(INPUT_CSV, encoding="ISO-8859-1").copy()
field.columns = [c.strip() for c in field.columns]

print("\nCSV columns:")
for c in field.columns:
    print(" -", c)

# ----------------------------
# Explicit schema handling (Date + Time)
# ----------------------------
if "lsu_Date" in field.columns:
    # combined_hypoxia_data.csv
    COL_DATE = "lsu_Date"

    if "lsu_Time CTD Cast began (UTC)" in field.columns:
        COL_TIME = "lsu_Time CTD Cast began (UTC)"
    elif "lsu_Time" in field.columns:
        COL_TIME = "lsu_Time"
    else:
        raise KeyError("Expected LSU time column not found in combined file.")

elif "Date" in field.columns:
    # lsu_hypoxia_2024.csv
    COL_DATE = "Date"

    if "Time CTD Cast began (UTC)" in field.columns:
        COL_TIME = "Time CTD Cast began (UTC)"
    else:
        raise KeyError("Expected time column not found in LSU file.")
else:
    raise KeyError("Expected 'Date' or 'lsu_Date' column not found.")

COL_STATION = "Station"
COL_LAT = "Lat"
COL_LON = "Long"
COL_CHL = "Surface Chl ( µg/L)"

print("Using date column:", COL_DATE)
print("Using time column:", COL_TIME)


def parse_field_datetime_utc(date_str: str, time_str: str) -> datetime:
    """
    Parse known LSU datetime formats (explicit, no guessing).
    """
    s = f"{date_str} {time_str}".strip()
    for fmt in ("%m/%d/%Y %H:%M", "%m/%d/%Y %H:%M:%S"):
        try:
            return datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    raise ValueError(f"Unrecognized datetime format: {s!r}")


field["field_time_utc"] = [
    parse_field_datetime_utc(d, t)
    for d, t in zip(field[COL_DATE].astype(str), field[COL_TIME].astype(str))
]
field["field_date_utc"] = field["field_time_utc"].dt.date

print("\nRows:", len(field))
print("Date range:", field["field_time_utc"].min(), "→", field["field_time_utc"].max())
display(field.head(3))


# ================================================================
# Compute bounding box around stations
# ================================================================
lat_min = float(field[COL_LAT].min())
lat_max = float(field[COL_LAT].max())
lon_min = float(field[COL_LON].min())
lon_max = float(field[COL_LON].max())

PAD = 1.0
bbox = (lon_min - PAD, lat_min - PAD, lon_max + PAD, lat_max + PAD)
print("\nBBox (W,S,E,N):", bbox)

plt.figure()
plt.scatter(field[COL_LON], field[COL_LAT], s=20)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Field stations")
plt.show()


# ================================================================
# Earthdata login
# ================================================================
if not os.environ.get("EARTHDATA_USERNAME") or not os.environ.get("EARTHDATA_PASSWORD"):
    raise RuntimeError("Set EARTHDATA_USERNAME and EARTHDATA_PASSWORD in environment.")

earthaccess.login(strategy="environment")


# ================================================================
# Discover PACE OCI Level-3 collections (robust via summary)
# ================================================================
collections = list(earthaccess.search_datasets(keyword="PACE OCI L3"))
print("\nCollections found:", len(collections))

rows = []
for c in collections[:200]:
    try:
        s = c.summary()
    except Exception:
        continue

    if isinstance(s, dict):
        rows.append({
            "ShortName": s.get("ShortName"),
            "Version": s.get("Version"),
            "EntryTitle": s.get("EntryTitle"),
            "ConceptId": s.get("concept-id"),
        })

df_col = pd.DataFrame(rows).dropna(subset=["ShortName"])

print("\ndf_col columns:", list(df_col.columns))
display(df_col.head(30))

if len(df_col) == 0:
    raise RuntimeError("No Level-3 PACE collections resolved from earthaccess.")


# ================================================================
# Choose a Level-3 collection
# ================================================================
CHOSEN_SHORTNAME = None
preferred_substrings = ["BGC", "CHL", "OC", "AOP", "RRS"]

for sub in preferred_substrings:
    m = df_col[df_col["ShortName"].astype(str).str.contains(sub, case=False, na=False)]
    if len(m) > 0:
        CHOSEN_SHORTNAME = m.iloc[0]["ShortName"]
        break

if CHOSEN_SHORTNAME is None:
    CHOSEN_SHORTNAME = df_col.iloc[0]["ShortName"]

print("\nCHOSEN_SHORTNAME:", CHOSEN_SHORTNAME)


# ================================================================
# Search Level-3 granules
# ================================================================
target_day = field["field_date_utc"].min()
t0 = datetime(target_day.year, target_day.month, target_day.day, tzinfo=timezone.utc)
t1 = datetime(target_day.year, target_day.month, target_day.day, 23, 59, 59, tzinfo=timezone.utc)

def search_l3(short_name, start_iso, end_iso, bbox_wsen):
    return list(
        earthaccess.search_data(
            short_name=short_name,
            temporal=(start_iso, end_iso),
            bounding_box=bbox_wsen,
        )
    )

results = search_l3(CHOSEN_SHORTNAME, t0.isoformat(), t1.isoformat(), bbox)

if len(results) == 0:
    month_start = datetime(target_day.year, target_day.month, 1, tzinfo=timezone.utc)
    month_end = (
        datetime(target_day.year + 1, 1, 1, tzinfo=timezone.utc)
        if target_day.month == 12
        else datetime(target_day.year, target_day.month + 1, 1, tzinfo=timezone.utc)
    ) - pd.Timedelta(seconds=1)

    print("No daily granules; trying month window")
    results = search_l3(CHOSEN_SHORTNAME, month_start.isoformat(), month_end.isoformat(), bbox)

print("Granules found:", len(results))
if not results:
    raise RuntimeError("No Level-3 granules found.")


# ================================================================
# Open granule + plot
# ================================================================
files = list(earthaccess.open(results[:1]))

try:
    ds = xr.open_dataset(files[0], engine="h5netcdf")
except Exception:
    ds = xr.open_dataset(files[0], engine="netcdf4")

print(ds)
print("\nVariables:", list(ds.variables)[:80])

CANDIDATES = ["chlor_a", "chl", "nflh", "kd_490", "bbp_443"]
PLOT_VAR = next((v for v in CANDIDATES if v in ds.variables), None)
if PLOT_VAR is None:
    raise RuntimeError("No expected variable found in Level-3 dataset.")

print("PLOT_VAR:", PLOT_VAR)


lat_name = "lat" if "lat" in ds.coords else "latitude"
lon_name = "lon" if "lon" in ds.coords else "longitude"

lats = ds[lat_name].values
lons = ds[lon_name].values

w, s, e, n = bbox
lon_pts = field[COL_LON].to_numpy()

if np.nanmax(lons) > 180:
    lon_pts = lon_pts % 360.0
    if w < 0:
        w, e = w % 360.0, e % 360.0

sub = ds[PLOT_VAR].sel(
    {
        lat_name: lats[(lats >= s) & (lats <= n)],
        lon_name: lons[(lons >= w) & (lons <= e)],
    }
)

plt.figure()
plt.pcolormesh(sub[lon_name], sub[lat_name], sub.values, shading="auto")
plt.colorbar(label=PLOT_VAR)
plt.scatter(lon_pts, field[COL_LAT], s=20)
plt.title(f"PACE L3 {PLOT_VAR} + stations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


# ================================================================
# Nearest-grid matchups + save
# ================================================================
def nearest_idx(arr, v):
    return int(np.argmin(np.abs(arr - v)))

vals = []
for _, r in field.iterrows():
    lat = float(r[COL_LAT])
    lon = float(r[COL_LON])
    if np.nanmax(lons) > 180 and lon < 0:
        lon = lon % 360.0

    vals.append(
        float(
            ds[PLOT_VAR].isel(
                {lat_name: nearest_idx(lats, lat), lon_name: nearest_idx(lons, lon)}
            ).values
        )
    )

field["pace_l3_nearest"] = vals

out_csv = OUTPUT_DIR / "pace_l3_quick_matchups_ralf.csv"
field[
    [COL_STATION, "field_time_utc", COL_LAT, COL_LON, COL_CHL, "pace_l3_nearest"]
].to_csv(out_csv, index=False)

print("\nWrote:", out_csv.resolve())


CWD: /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel
Home: /home/jovyan

Listing / (first 30):
 - /bin
 - /boot
 - /dev
 - /etc
 - /home
 - /lib
 - /lib64
 - /media
 - /mnt
 - /opt
 - /proc
 - /root
 - /run
 - /sbin
 - /srv
 - /sys
 - /tmp
 - /usr
 - /var
 - /bin.usr-is-merged
 - /init
 - /lib.usr-is-merged
 - /libexec
 - /rocker_scripts
 - /pyrocket_scripts

Listing /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel (first 30):
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel/tm1_notebook.ipynb
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel/pace_matchups.ipynb
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel/pace_l3_matchup.ipynb
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel/test_get_pace_data.ipynb
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ralf Riedel/.ipynb_checkpoints
 - /home/jovyan/2026-proj-hypoxia-zones/contributor_folders/Ral

KeyError: 'Expected LSU time column not found in combined file.'